# Step 4b on Kaggle: train Audio2Motion (configuration v2)

1. Create a private Kaggle dataset containing the project code (`tinypt/`, `scripts/`, `pyproject.toml`) and `data/vocaset_motion/vocaset_motion.npz`.
2. Import this notebook, add the dataset as input, and enable a GPU accelerator and internet.
3. Run all cells (about 15-25 min) and unzip `04b_audio2motion_v2_better_jaw_correlation.zip` from the output panel into `outputs/flame_pipeline/04b_audio2motion_v2_better_jaw_correlation/`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob
import os

NPZ = glob.glob('/kaggle/input/**/vocaset_motion.npz', recursive=True)[0]
CODE = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/tinypt/__init__.py', recursive=True)[0]))
OUT = '/kaggle/working/04b_audio2motion_v2_better_jaw_correlation'
print(NPZ, CODE, sep='
')

In [ ]:
!pip -q install transformers

In [ ]:
# v2 = the script defaults (60 epochs, speed perturbation, time masking, motion-spread loss)
!PYTHONPATH="$CODE" python "$CODE/scripts/flame_pipeline/04b_train_audio2motion.py" --data "$NPZ" --out "$OUT" --feature wav2vec2

In [ ]:
import json
from IPython.display import Image, display

print(json.dumps(json.load(open(f'{OUT}/metrics.json'))['test'], indent=2))
display(Image(f'{OUT}/01_training_and_jaw.png'))
display(Image(f'{OUT}/02_test_lve.png'))

In [ ]:
# The cached audio features are large and re-creatable, so they are left out of the zip.
import zipfile

with zipfile.ZipFile(f'{OUT}.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUT):
        if not f.startswith('features_'):
            z.write(os.path.join(OUT, f), f)